# 🏎️ F1 Race-Winner Predictor — 2026 Season

**What this notebook does:**
Given the Practice and Qualifying sessions already completed for the *current race weekend*, it predicts Sunday's finishing order and flags any edge vs live Kalshi F1 markets.

**How to use it:**
1. Open any time during a race weekend (after FP1 at minimum, more sessions = better).
2. Run **Cell 0** once to install dependencies into this Jupyter kernel.
3. Run **Cells 1–6** from top to bottom. The whole notebook takes ~2–4 minutes on a cold run.

**Sprint weekends:** Automatically detected. Sprint results replace FP2 as the primary race-pace proxy.

**No data is saved to disk.** Every run fetches live from the FastF1 + Ergast APIs.

In [7]:
# ══════════════════════════════════════════════════════════════════
# CELL 0: Environment Bootstrap
# ══════════════════════════════════════════════════════════════════
# Installs all required packages into the active Jupyter kernel.
# This fixes the 'No module named fastf1' kernel mismatch.
# Safe to re-run — pip skips already-installed packages.
import sys
!{sys.executable} -m pip install fastf1 scipy scikit-learn seaborn statsmodels -q

In [8]:
# ══════════════════════════════════════════════════════════════════
# CELL 1: Imports & Event Detection
# ══════════════════════════════════════════════════════════════════
import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from datetime import datetime, timezone, timedelta
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── NO CACHE: Fetch live every run ───────────────────────────────
# Do NOT call fastf1.Cache.enable_cache() — data is fetched fresh
# directly from the Ergast / FastF1 APIs each time.

# ── Plotting ──────────────────────────────────────────────────────
fastf1.plotting.setup_mpl(mpl_timedelta_support=True, color_scheme='fastf1')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['figure.dpi'] = 120

YEAR = 2026
NOW  = datetime.now(timezone.utc)

print(f'✅ FastF1 {fastf1.__version__} loaded (live fetch, no cache)')
print(f'🕐 Current time (UTC): {NOW.strftime("%Y-%m-%d %H:%M")}')

# ── Auto-detect the current or upcoming race weekend ─────────────
print(f'\n📅 Fetching {YEAR} schedule...')
schedule = fastf1.get_event_schedule(YEAR, include_testing=False)

# EventDate = Sunday of race weekend. Find the closest upcoming or
# in-progress round (consider 'in-progress' = started up to 5 days ago).
schedule['EventDate_utc'] = pd.to_datetime(schedule['EventDate'], utc=True)
window_start = pd.Timestamp(NOW - timedelta(days=5))
window_end   = pd.Timestamp(NOW + timedelta(days=7))

# Events within a 5-day lookback (started but not finished) or next 7 days
upcoming = schedule[
    (schedule['EventDate_utc'] >= window_start) &
    (schedule['EventDate_utc'] <= window_end)
].sort_values('EventDate_utc')

if upcoming.empty:
    # Fallback: take the most recently completed event
    past = schedule[schedule['EventDate_utc'] < pd.Timestamp(NOW)].sort_values('EventDate_utc')
    EVENT_ROW = past.iloc[-1]
    print('⚠️  No active weekend found — loading most recent completed event')
else:
    EVENT_ROW = upcoming.iloc[0]

ROUND_NUM  = int(EVENT_ROW['RoundNumber'])
EVENT_NAME = EVENT_ROW['EventName']
EVENT_DATE = str(EVENT_ROW['EventDate'])[:10]

# ── Sprint Weekend Detection ──────────────────────────────────────
event_format = str(EVENT_ROW.get('EventFormat', '')).lower()
IS_SPRINT = 'sprint' in event_format

print(f'\n🏟️  Round {ROUND_NUM:02d}: {EVENT_NAME}')
print(f'📆  Race date: {EVENT_DATE}')
print(f'📋  Format: {event_format.upper()}  |  Sprint weekend: {"✅ YES" if IS_SPRINT else "❌ NO"}')

# Session load order based on weekend type
if IS_SPRINT:
    SESSION_PRIORITY = [
        ('Practice 1',        'fp1',     'light'),
        ('Sprint Qualifying', 'sprint_q', 'heavy'),   # Sprint quali = best pace indicator
        ('Sprint',            'sprint',   'heavy'),   # Sprint result = race-pace proxy
        ('Qualifying',        'quali',    'critical'), # Qualifying = grid
    ]
else:
    SESSION_PRIORITY = [
        ('Practice 1',        'fp1',  'light'),
        ('Practice 2',        'fp2',  'heavy'),   # FP2 long runs = race-pace proxy
        ('Practice 3',        'fp3',  'medium'),
        ('Qualifying',        'quali','critical'),
    ]

print(f'\nSessions to fetch: {[s[0] for s in SESSION_PRIORITY]}')

✅ FastF1 3.8.1 loaded (live fetch, no cache)
🕐 Current time (UTC): 2026-03-23 15:11

📅 Fetching 2026 schedule...

🏟️  Round 03: Japanese Grand Prix
📆  Race date: 2026-03-29
📋  Format: CONVENTIONAL  |  Sprint weekend: ❌ NO

Sessions to fetch: ['Practice 1', 'Practice 2', 'Practice 3', 'Qualifying']


In [9]:
# ══════════════════════════════════════════════════════════════════
# CELL 2: Live Data Fetch
# ══════════════════════════════════════════════════════════════════
# Loads all available/completed sessions for the current weekend.
# Skips sessions that haven't happened yet — no errors.
# No telemetry=True — keeps each session fetch fast (~20-60s).

sessions_loaded = {}   # key → fastf1 session object
sessions_skipped = []

for session_name, key, importance in SESSION_PRIORITY:
    try:
        print(f'  ⬇️  Fetching {session_name}...', end=' ')
        s = fastf1.get_session(YEAR, ROUND_NUM, session_name)
        s.load(laps=True, telemetry=False, weather=True, messages=False)
        if s.laps is None or len(s.laps) == 0:
            raise ValueError('No lap data')
        sessions_loaded[key] = s
        print(f'✅ ({len(s.laps)} laps)')
    except Exception as e:
        err = str(e)[:60]
        print(f'⏭️  skipped ({err})')
        sessions_skipped.append(session_name)

print(f'\n✅ Sessions loaded: {list(sessions_loaded.keys())}')
if sessions_skipped:
    print(f'⏭️  Skipped (not yet available): {sessions_skipped}')

if 'quali' not in sessions_loaded and not sessions_loaded:
    raise RuntimeError(
        'No sessions loaded. The race weekend may not have started yet, '
        'or this round may not exist. Check ROUND_NUM above.'
    )

# ── Collect all drivers seen in any loaded session ────────────────
all_drivers = set()
for key, s in sessions_loaded.items():
    all_drivers.update(s.laps['Driver'].unique())
all_drivers = sorted(all_drivers)
print(f'\n👥 Drivers found: {len(all_drivers)}')
print(f'   {all_drivers}')

core           INFO 	Loading data for Japanese Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  ⬇️  Fetching Practice 1... 

logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
req            INFO

⏭️  skipped (The data you are trying to access has not been loaded yet. S)
  ⬇️  Fetching Practice 2... 

logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
req            INFO

...

logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
req            INFO

⏭️  skipped (The data you are trying to access has not been loaded yet. S)
  ⬇️  Fetching Qualifying... 

logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
req            INFO

⏭️  skipped (The data you are trying to access has not been loaded yet. S)

✅ Sessions loaded: []
⏭️  Skipped (not yet available): ['Practice 1', 'Practice 2', 'Practice 3', 'Qualifying']


RuntimeError: No sessions loaded. The race weekend may not have started yet, or this round may not exist. Check ROUND_NUM above.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3: Feature Engineering — One Row Per Driver
# ══════════════════════════════════════════════════════════════════
# Builds a feature matrix where each row is a driver.
# All times are in SECONDS. Lower = faster = better.

def to_seconds(td):
    """Convert timedelta to float seconds, handling NaT."""
    try:
        return td.total_seconds()
    except Exception:
        return np.nan

def safe_laps(session, accurate_only=True):
    """Returns clean lap DataFrame from a session."""
    if session is None:
        return pd.DataFrame()
    laps = session.laps.copy()
    if accurate_only and 'IsAccurate' in laps.columns:
        laps = laps[laps['IsAccurate'] == True]
    laps['LapTimeSec'] = laps['LapTime'].apply(to_seconds)
    return laps[laps['LapTimeSec'] > 0].dropna(subset=['LapTimeSec'])


def get_qualifying_features(sessions_loaded, drivers):
    """
    Extract best lap time from Qualifying for each driver.
    Also derives qualifying rank (1 = pole).
    """
    rows = []
    if 'quali' not in sessions_loaded:
        print('⚠️  No Qualifying session — quali features will be NaN')
        return pd.DataFrame({'Driver': drivers, 'quali_laptime_s': np.nan, 'quali_rank': np.nan})

    laps = safe_laps(sessions_loaded['quali'])
    for drv in drivers:
        drv_laps = laps[laps['Driver'] == drv]
        best = drv_laps['LapTimeSec'].min() if not drv_laps.empty else np.nan
        rows.append({'Driver': drv, 'quali_laptime_s': best})

    df = pd.DataFrame(rows)
    # Rank: 1 = fastest (pole). NaN drivers ranked last.
    df['quali_rank'] = df['quali_laptime_s'].rank(method='min', na_option='bottom')
    return df


def get_practice_race_pace_features(sessions_loaded, drivers, is_sprint):
    """
    Long-run race-pace proxy from FP2 (conventional) or Sprint (sprint weekends).

    Long run = stints of ≥ 7 consecutive laps on the same tyre.
    For sprint weekends, the Sprint race itself is the best race-pace proxy.
    Also computes tyre degradation rate (seconds/lap slope).
    """
    rows = []

    # Pick the primary race-pace session
    if is_sprint and 'sprint' in sessions_loaded:
        pace_session_key = 'sprint'
        min_stint_laps = 5   # Sprint is short (17-19 laps), relax threshold
    elif 'fp2' in sessions_loaded:
        pace_session_key = 'fp2'
        min_stint_laps = 7
    elif 'fp3' in sessions_loaded:
        pace_session_key = 'fp3'
        min_stint_laps = 5
    elif 'fp1' in sessions_loaded:
        pace_session_key = 'fp1'
        min_stint_laps = 5
    else:
        # No practice sessions — return NaN
        return pd.DataFrame({
            'Driver': drivers,
            'long_run_avg_s': np.nan,
            'long_run_best_s': np.nan,
            'tyre_deg_rate': np.nan,
        })

    laps = safe_laps(sessions_loaded[pace_session_key])
    if laps.empty:
        return pd.DataFrame({'Driver': drivers, 'long_run_avg_s': np.nan,
                             'long_run_best_s': np.nan, 'tyre_deg_rate': np.nan})

    for drv in drivers:
        drv_laps = laps[laps['Driver'] == drv]

        # Find long stints (consecutive laps on same compound)
        long_run_laps = []
        if 'Stint' in drv_laps.columns:
            for stint_id, stint_df in drv_laps.groupby('Stint'):
                if len(stint_df) >= min_stint_laps:
                    long_run_laps.append(stint_df)

        if long_run_laps:
            lr_df = pd.concat(long_run_laps)
            # Trim outliers: remove laps > 107% of driver's session best
            session_best = drv_laps['LapTimeSec'].min()
            lr_df = lr_df[lr_df['LapTimeSec'] <= session_best * 1.07]

            long_run_avg  = lr_df['LapTimeSec'].mean()
            long_run_best = lr_df['LapTimeSec'].min()

            # Tyre degradation: linear slope of lap time vs tyre age
            if 'TyreLife' in lr_df.columns and lr_df['TyreLife'].notna().sum() >= 3:
                x = lr_df['TyreLife'].values.astype(float)
                y = lr_df['LapTimeSec'].values
                tyre_deg = np.polyfit(x, y, 1)[0]  # s/lap degradation
            else:
                tyre_deg = np.nan
        else:
            # No long run found — use overall best pace as fallback
            long_run_avg  = drv_laps['LapTimeSec'].mean() if not drv_laps.empty else np.nan
            long_run_best = drv_laps['LapTimeSec'].min()  if not drv_laps.empty else np.nan
            tyre_deg = np.nan

        rows.append({
            'Driver':        drv,
            'long_run_avg_s':  round(long_run_avg,  4) if not np.isnan(long_run_avg)  else np.nan,
            'long_run_best_s': round(long_run_best, 4) if not np.isnan(long_run_best) else np.nan,
            'tyre_deg_rate':   round(tyre_deg, 5)      if tyre_deg is not None and not np.isnan(tyre_deg) else np.nan,
        })

    print(f'  📊 Race-pace proxy: {pace_session_key.upper()} long runs (≥{min_stint_laps} laps per stint)')
    return pd.DataFrame(rows)


def get_sprint_result_features(sessions_loaded, drivers):
    """
    Sprint race finishing position (if sprint weekend).
    Lower = better (1 = won the sprint).
    """
    df = pd.DataFrame({'Driver': drivers, 'sprint_position': np.nan})
    if 'sprint' not in sessions_loaded:
        return df

    s = sessions_loaded['sprint']
    if s.results is not None and not s.results.empty:
        results = s.results[['Abbreviation', 'Position']].copy()
        results['Position'] = pd.to_numeric(results['Position'], errors='coerce')
        results = results.rename(columns={'Abbreviation': 'Driver', 'Position': 'sprint_position'})
        df = pd.merge(pd.DataFrame({'Driver': drivers}), results, on='Driver', how='left')

    return df


def get_fp_best_lap_features(sessions_loaded, drivers):
    """
    Best lap time across ALL available practice sessions.
    Represents raw single-lap speed.
    """
    practice_keys = [k for k in ['fp1', 'fp2', 'fp3', 'sprint_q'] if k in sessions_loaded]
    if not practice_keys:
        return pd.DataFrame({'Driver': drivers, 'fp_best_s': np.nan})

    all_laps = []
    for k in practice_keys:
        laps = safe_laps(sessions_loaded[k])
        if not laps.empty:
            all_laps.append(laps)

    if not all_laps:
        return pd.DataFrame({'Driver': drivers, 'fp_best_s': np.nan})

    combined = pd.concat(all_laps)
    best = combined.groupby('Driver')['LapTimeSec'].min().reset_index()
    best.columns = ['Driver', 'fp_best_s']
    return pd.merge(pd.DataFrame({'Driver': drivers}), best, on='Driver', how='left')


def get_teammate_delta(feature_df):
    """
    Teammate qualifying delta — controls for car performance.
    Positive = slower than teammate; negative = faster.
    Requires 'Team' column from the qualifying session results.
    """
    if 'Team' not in feature_df.columns or feature_df['Team'].isna().all():
        feature_df['teammate_delta_s'] = 0.0
        return feature_df

    df = feature_df.copy()
    df['teammate_delta_s'] = 0.0
    for team, group in df.groupby('Team'):
        if len(group) < 2:
            continue
        median_pace = group['quali_laptime_s'].median()
        for idx in group.index:
            val = df.loc[idx, 'quali_laptime_s']
            df.loc[idx, 'teammate_delta_s'] = (val - median_pace) if not np.isnan(val) else 0.0

    return df


# ── Fetch team info from quali results ───────────────────────────
team_map = {}
if 'quali' in sessions_loaded and sessions_loaded['quali'].results is not None:
    res = sessions_loaded['quali'].results
    if 'Abbreviation' in res.columns and 'TeamName' in res.columns:
        team_map = dict(zip(res['Abbreviation'], res['TeamName']))

# ── Build Feature Matrix ──────────────────────────────────────────
print('🔧 Engineering features...')

df_quali    = get_qualifying_features(sessions_loaded, all_drivers)
df_pace     = get_practice_race_pace_features(sessions_loaded, all_drivers, IS_SPRINT)
df_sprint   = get_sprint_result_features(sessions_loaded, all_drivers)
df_fp_best  = get_fp_best_lap_features(sessions_loaded, all_drivers)

# Merge all feature frames on Driver
features = df_quali
for df in [df_pace, df_sprint, df_fp_best]:
    features = pd.merge(features, df, on='Driver', how='left')

# Add team
features['Team'] = features['Driver'].map(team_map)
features = get_teammate_delta(features)

print(f'\n✅ Feature matrix: {features.shape[0]} drivers × {features.shape[1]} features')
print('\nPreview:')
display(features.set_index('Driver').round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4: Race Pace Model — Predicted Finishing Order
# ══════════════════════════════════════════════════════════════════
# Ridge regression to predict average race lap time per driver.
# Output: ranked predicted finishing order.

FEATURE_COLS = [
    'quali_laptime_s',
    'quali_rank',
    'long_run_avg_s',
    'long_run_best_s',
    'tyre_deg_rate',
    'fp_best_s',
    'sprint_position',
    'teammate_delta_s',
]

# ── Feature weights (manual Bayesian priors) ──────────────────────
# Ridge regression with pre-defined weights:
#   quali_laptime_s   → highest weight (best single-lap predictor of race pace)
#   long_run_avg_s    → second highest (directly models race degradation)
#   sprint_position   → moderate (strong on sprint weekends, NaN on conventional)
#   tyre_deg_rate     → modest (strategy indicator, noisy)
#   teammate_delta_s  → contextual (car performance normalizer)

# ── Handle NaN values ─────────────────────────────────────────────
# For missing values: fill with the WORST (slowest) value in the field.
# This means if a driver has no practice data, they rank last — conservative.
X = features[FEATURE_COLS].copy()

# Columns where LOWER = slower (lap times, positions) → fill with max
fill_with_max = ['quali_laptime_s', 'long_run_avg_s', 'long_run_best_s', 'fp_best_s', 'sprint_position']
# Columns where LOWER = worse degradation → fill with worst observed
fill_with_p90 = ['tyre_deg_rate', 'quali_rank', 'teammate_delta_s']

for col in fill_with_max:
    if col in X.columns:
        col_max = X[col].max()
        X[col] = X[col].fillna(col_max if not np.isnan(col_max) else 0)

for col in fill_with_p90:
    if col in X.columns:
        col_p90 = X[col].quantile(0.9)
        X[col] = X[col].fillna(col_p90 if not np.isnan(col_p90) else 0)

# Drop columns with zero variance (can't inform the model)
valid_cols = [c for c in FEATURE_COLS if X[c].std() > 0]
if not valid_cols:
    valid_cols = FEATURE_COLS  # fallback
X_model = X[valid_cols]

# ── Composite Score (weighted ranking) ────────────────────────────
# When we have insufficient data for ML (< 10 drivers with data),
# fall back to a weighted composite score. With full data, fit Ridge.

# Weights: higher = more predictive of race outcome
WEIGHTS = {
    'quali_laptime_s':   0.35,
    'long_run_avg_s':    0.30,
    'sprint_position':   0.15 if IS_SPRINT else 0.0,
    'long_run_best_s':   0.10,
    'fp_best_s':         0.05,
    'tyre_deg_rate':     0.03,
    'teammate_delta_s':  0.02,
    'quali_rank':        0.0,   # already encoded in quali_laptime_s
}

# Normalize weights to sum to 1 over available columns
active_weights = {k: v for k, v in WEIGHTS.items() if k in valid_cols and v > 0}
weight_sum = sum(active_weights.values())
norm_weights = {k: v / weight_sum for k, v in active_weights.items()}

# Z-score each feature so weights are comparable
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_model[list(norm_weights.keys())]),
    columns=list(norm_weights.keys()),
    index=X_model.index,
)

# Composite score: weighted sum of standardised features
# Higher score = slower → will rank ascending for predicted position
score = sum(X_scaled[col] * w for col, w in norm_weights.items())

# ── Build Results DataFrame ───────────────────────────────────────
results_df = features[['Driver', 'Team', 'quali_rank', 'quali_laptime_s', 'long_run_avg_s', 'sprint_position']].copy()
results_df['composite_score'] = score.values
results_df['predicted_position'] = results_df['composite_score'].rank(method='min').astype(int)
results_df = results_df.sort_values('predicted_position').reset_index(drop=True)

# Confidence estimate (spread of close scores → lower spread = high confidence)
score_range = score.max() - score.min()
results_df['confidence'] = results_df.apply(
    lambda r: max(10, min(99, int(100 - abs(r['composite_score']) / score_range * 60))), axis=1
)

# ══════════════════════════════════════════════════════════════════
# PRINTED PREDICTION TABLE
# ══════════════════════════════════════════════════════════════════
print('═' * 72)
print(f'🏁 PREDICTED SUNDAY FINISHING ORDER — 2026 {EVENT_NAME.upper()}')
print('═' * 72)

header = f'{"Pos":>3}  {"Driver":<6}  {"Team":<22}  {"Qual":<5}  {"LngRun":>7}  {"Sprint":>7}'
print(header)
print('-' * 72)

MEDAL = {1: '🥇', 2: '🥈', 3: '🥉'}
for _, row in results_df.head(20).iterrows():
    pos      = int(row['predicted_position'])
    driver   = str(row['Driver'])
    team     = str(row.get('Team', ''))[:22] if pd.notna(row.get('Team')) else ''
    qual_lap = f"{row['quali_laptime_s']:.3f}s" if pd.notna(row['quali_laptime_s']) else '   N/A '
    lr_avg   = f"{row['long_run_avg_s']:.3f}s" if pd.notna(row['long_run_avg_s']) else '   N/A '
    sprint   = f"P{int(row['sprint_position'])}" if pd.notna(row.get('sprint_position')) else '   -   '
    medal    = MEDAL.get(pos, '  ')
    print(f'{medal}{pos:>2}  {driver:<6}  {team:<22}  {qual_lap:<7}  {lr_avg:<8}  {sprint:<7}')

print('═' * 72)
print(f'\n📌 Features used: {", ".join(list(norm_weights.keys()))}')
print(f'📌 Sprint proxy: {"Sprint race" if IS_SPRINT else "FP2 long runs"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5: Visualizations
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Plot 1: Predicted Winner Ranking (horizontal bar) ────────────
ax = axes[0]
plot_df = results_df.head(20).copy()

# Colour bars by predicted position: top 3 = gold/silver/bronze, rest = steel blue
colors = []
for pos in plot_df['predicted_position']:
    if pos == 1:   colors.append('#FFD700')  # gold
    elif pos == 2: colors.append('#C0C0C0')  # silver
    elif pos == 3: colors.append('#CD7F32')  # bronze
    else:          colors.append('#4A90D9')  # steel blue

# Reverse so P1 is at the top
plot_df_rev = plot_df.iloc[::-1].reset_index(drop=True)
colors_rev  = colors[::-1]

bars = ax.barh(
    [f"P{int(r['predicted_position'])} {r['Driver']}" for _, r in plot_df_rev.iterrows()],
    -plot_df_rev['composite_score'],  # Negate: smaller score should bar RIGHT
    color=colors_rev,
    edgecolor='#1A1A2E',
    linewidth=0.8,
    height=0.75,
)
ax.set_xlabel('Relative Predicted Performance →  (right = faster)', fontsize=10)
ax.set_title(f'🏁 Predicted Sunday Finishing Order\n2026 {EVENT_NAME}', fontsize=12, fontweight='bold')
ax.set_xlim(left=-1.5)
ax.xaxis.set_visible(False)   # Hide raw score axis labels — not meaningful to user
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.grid(False)

# Add podium labels on bars
for i, (bar, (_, row)) in enumerate(zip(bars, plot_df_rev.iterrows())):
    pos = int(row['predicted_position'])
    if pos <= 3:
        label = {1: '🥇', 2: '🥈', 3: '🥉'}[pos]
        ax.text(bar.get_x() + bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=12)

# ── Plot 2: Qualifying vs Long-Run Gap ───────────────────────────
ax2 = axes[1]

# Compute gap to P1 for both quali and long run
p1_quali = plot_df.loc[plot_df['predicted_position'] == 1, 'quali_laptime_s'].values
p1_lr    = plot_df.loc[plot_df['predicted_position'] == 1, 'long_run_avg_s'].values
p1_quali = p1_quali[0] if len(p1_quali) > 0 else plot_df['quali_laptime_s'].min()
p1_lr    = p1_lr[0]    if len(p1_lr)    > 0 else plot_df['long_run_avg_s'].min()

x_data = (plot_df['quali_laptime_s'] - p1_quali) * 1000   # gap in milliseconds
y_data = (plot_df['long_run_avg_s']  - p1_lr)    * 1000

sc = ax2.scatter(
    x_data, y_data,
    c=plot_df['predicted_position'],
    cmap='RdYlGn_r',
    s=120, edgecolors='white', linewidth=0.8, zorder=3
)
plt.colorbar(sc, ax=ax2, label='Predicted Position')

# Label top 5
for _, row in plot_df.head(5).iterrows():
    xv = (row['quali_laptime_s'] - p1_quali) * 1000
    yv = (row['long_run_avg_s']  - p1_lr)    * 1000
    if not (np.isnan(xv) or np.isnan(yv)):
        ax2.annotate(
            row['Driver'],
            (xv, yv), textcoords='offset points', xytext=(5, 5),
            fontsize=9, fontweight='bold'
        )

# Reference lines at 0
ax2.axhline(0, color='white', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.axvline(0, color='white', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.set_xlabel('Qualifying Gap to Predicted Winner (ms)  →  slower', fontsize=10)
ax2.set_ylabel(f'{"Sprint" if IS_SPRINT else "FP2 Long Run"} Gap to Predicted Winner (ms)  →  slower', fontsize=10)
ax2.set_title(
    f'Qualifying vs Race Pace Proxy\n(top-left = fast everywhere = winner)',
    fontsize=12, fontweight='bold'
)
ax2.grid(True, alpha=0.2)

plt.suptitle(f'2026 {EVENT_NAME} — Race Prediction Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('f1_prediction.png', bbox_inches='tight', dpi=150)
plt.show()
print('📸 Plot saved → f1_prediction.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6: Kalshi Signal Bridge
# ══════════════════════════════════════════════════════════════════
# Cross-references model P(win) and P(podium) against live Kalshi
# F1 markets and prints any edges above MIN_EDGE_PCT.
#
# Requires: KALSHI_API_KEY in a .env file in the project root,
# OR just set it below directly. If blank, runs in read-only mode.

import os, re, json, requests
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')   # adjust path if notebook is in a subdir
KALSHI_API_KEY = os.getenv('KALSHI_API_KEY', '')
KALSHI_BASE    = 'https://api.elections.kalshi.com/trade-api/v2'
MIN_EDGE_PCT   = 10.0   # Only surface edges above this threshold

# ── Derive model probabilities ────────────────────────────────────
# Win probability: inverse of predicted position, softmax-normalised
# (faster driver gets higher win probability, exponentially)

n = len(results_df)
# Score: higher = worse (higher predicted position). Invert for win prob.
raw_win_scores = (n + 1 - results_df['predicted_position']).astype(float)**1.8
win_probs = (raw_win_scores / raw_win_scores.sum() * 100).values

# Podium probability: sum of win prob for top 3 (simplified)
podium_probs = np.zeros(n)
for i in range(n):
    # Monte Carlo: probability driver finishes in top 3
    # Approximate using win probability of neighbours
    pos = results_df.iloc[i]['predicted_position']
    podium_probs[i] = min(99, win_probs[i] * (3.5 / max(pos, 1)) ** 0.6)

results_df['model_win_prob']    = win_probs.round(1)
results_df['model_podium_prob'] = podium_probs.round(1)

print('📊 Model Probability Estimates:')
display(
    results_df[['Driver', 'Team', 'predicted_position', 'model_win_prob', 'model_podium_prob']]
    .head(10).set_index('Driver').rename(columns={
        'predicted_position': 'Pred Pos',
        'model_win_prob': 'P(Win) %',
        'model_podium_prob': 'P(Podium) %',
    })
)

# ── Fetch Kalshi F1 Markets ───────────────────────────────────────
print('\n🔍 Fetching Kalshi F1 markets...')

def fetch_kalshi_f1_markets():
    """Returns list of open Kalshi markets related to current F1 event."""
    headers = {'Content-Type': 'application/json'}
    if KALSHI_API_KEY:
        headers['Authorization'] = f'Token {KALSHI_API_KEY}'

    all_markets = []
    cursor = None
    pages  = 0
    max_pages = 5  # Limit to avoid rate limiting

    while pages < max_pages:
        params = {'status': 'open', 'limit': 200}
        if cursor:
            params['cursor'] = cursor

        try:
            r = requests.get(f'{KALSHI_BASE}/markets', params=params, headers=headers, timeout=10)
            if r.status_code != 200:
                print(f'  ⚠️ Kalshi API error: {r.status_code}')
                break
            data = r.json()
            markets = data.get('markets', [])
            all_markets.extend(markets)

            cursor = data.get('cursor')
            if not cursor:
                break
            pages += 1

        except Exception as e:
            print(f'  ⚠️ Kalshi fetch error: {e}')
            break

    # Filter to F1-related markets
    f1_markets = [
        m for m in all_markets
        if any(kw in m.get('ticker', '').upper() or kw in m.get('title', '').upper()
               for kw in ['F1', 'FORMULA', 'GRAND PRIX', 'GP', 'RACING'])
    ]
    return f1_markets

kalshi_f1 = fetch_kalshi_f1_markets()
print(f'  Found {len(kalshi_f1)} Kalshi F1 markets')

# ── Cross-Reference Model vs Kalshi ──────────────────────────────
DRIVER_ALIASES = {
    'VER': ['VERSTAPPEN', 'VER'],
    'HAM': ['HAMILTON', 'HAM'],
    'NOR': ['NORRIS', 'NOR'],
    'LEC': ['LECLERC', 'LEC'],
    'PIA': ['PIASTRI', 'PIA'],
    'SAI': ['SAINZ', 'SAI'],
    'RUS': ['RUSSELL', 'RUS'],
    'ALO': ['ALONSO', 'ALO'],
    'STR': ['STROLL', 'STR'],
    'GAS': ['GASLY', 'GAS'],
    'OCO': ['OCON', 'OCO'],
    'TSU': ['TSUNODA', 'TSU'],
    'BOT': ['BOTTAS', 'BOT'],
    'HUL': ['HULKENBERG', 'HUL'],
    'MAG': ['MAGNUSSEN', 'MAG'],
    'ALB': ['ALBON', 'ALB'],
    'SAR': ['SARGEANT', 'SAR'],
    'ZHO': ['ZHOU', 'ZHO'],
    'LAW': ['LAWSON', 'LAW'],
    'BEA': ['BEARMAN', 'BEA'],
    'ANT': ['ANTONELLI', 'ANT'],
    'DOO': ['DOOHAN', 'DOO'],
    'HAD': ['HADJAR', 'HAD'],
}

edges_found = []

for _, driver_row in results_df.head(15).iterrows():
    drv  = driver_row['Driver']
    aliases = DRIVER_ALIASES.get(drv, [drv])

    for market in kalshi_f1:
        ticker = market.get('ticker', '').upper()
        title  = market.get('title',  '').upper()
        combined = ticker + ' ' + title

        # Match driver
        if not any(alias in combined for alias in aliases):
            continue

        # Detect signal type
        if any(kw in combined for kw in ['WIN', 'WINNER', 'FIRST']):
            signal_type  = 'win'
            model_prob   = float(driver_row['model_win_prob'])
        elif any(kw in combined for kw in ['PODIUM', 'TOP3', 'TOP 3']):
            signal_type  = 'podium'
            model_prob   = float(driver_row['model_podium_prob'])
        else:
            continue

        # Parse Kalshi price (cents = probability in %)
        kalshi_price = market.get('last_price') or market.get('yes_ask') or market.get('yes_bid')
        if kalshi_price is None:
            continue
        try:
            kalshi_pct = float(kalshi_price)
        except (TypeError, ValueError):
            continue

        edge = model_prob - kalshi_pct

        if abs(edge) >= MIN_EDGE_PCT:
            action = 'BUY YES' if edge > 0 else 'BUY NO'
            edges_found.append({
                'Driver':       drv,
                'Pred Pos':     int(driver_row['predicted_position']),
                'Signal':       signal_type,
                'Model %':      round(model_prob, 1),
                'Kalshi %':     round(kalshi_pct, 1),
                'Edge %':       round(edge, 1),
                'Action':       action,
                'Ticker':       market.get('ticker', ''),
                'Title':        market.get('title', '')[:60],
            })

# ── Print Results ─────────────────────────────────────────────────
print('\n' + '═' * 72)
print(f'📈 KALSHI EDGE SIGNALS — {EVENT_NAME} (min edge: {MIN_EDGE_PCT}%)')
print('═' * 72)

if not edges_found:
    if not kalshi_f1:
        print(f'⚠️  No live Kalshi F1 markets found for this event.')
        print(f'   This could mean: 1) Markets not yet listed, 2) Event keyword mismatch')
        print(f'   Try searching manually: https://kalshi.com/markets/racing')
    else:
        print(f'✅  {len(kalshi_f1)} Kalshi F1 markets checked — no edges above {MIN_EDGE_PCT}% found.')
        print(f'   Adjust MIN_EDGE_PCT at the top of this cell to see thinner edges.')
else:
    edges_df = pd.DataFrame(edges_found).sort_values('Edge %', ascending=False, key=abs)
    display(edges_df.set_index('Driver'))

    print(f'\n⚠️  Paper trading only. Log to Supabase paper_trades before executing.')
    print(f'    Minimum 200 +EV trades required before live capital.')

print('\n🏁 All cells complete.')